In [1]:
!pip install wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 36.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Imports**


In [3]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cuda


**Reading datasets**

In [4]:
train_df = pd.read_csv("/content/drive/MyDrive/WalmartPrices/NBeats/train_parquet.csv")
val_df = pd.read_csv("/content/drive/MyDrive/WalmartPrices/NBeats/val_parquet.csv")


In [7]:
print(train_df.columns.to_list())

['Store', 'Dept', 'Date', 'y', 'IsHoliday', 'unique_id', 'time_idx']


**WandB**

In [6]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tdola23 (tdola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

**Baseline**

In [8]:
import numpy as np

def wmae(y_true, y_pred, is_holiday):
    weights = is_holiday.map({True: 5, False: 1})
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

last known value

In [9]:
last_vals = train_df.sort_values('time_idx').groupby('unique_id')['y'].last()

naive_eval = val_df.merge(last_vals.rename('naive_pred'), on='unique_id', how='left')

missing_naive = naive_eval['naive_pred'].isna().sum()
print("Rows with no naive prediction:", missing_naive)

global_mean = train_df['y'].mean()
naive_eval['naive_pred'] = naive_eval['naive_pred'].fillna(global_mean)

naive_wmae = wmae(naive_eval['y'], naive_eval['naive_pred'], naive_eval['IsHoliday'])
print("Naive WMAE:", naive_wmae)

Rows with no naive prediction: 106
Naive WMAE: 2754.5248016741434


Seasonal Baseline

In [10]:
seasonal_lookup = train_df.set_index(['unique_id', 'time_idx'])['y']

val_df_seasonal = val_df.copy()
val_df_seasonal['seasonal_time_idx'] = val_df_seasonal['time_idx'] - 52

seasonal_eval = val_df_seasonal.merge(
    seasonal_lookup.rename('seasonal_pred'),
    left_on=['unique_id', 'seasonal_time_idx'],
    right_index=True,
    how='left'
)

missing_seasonal = seasonal_eval['seasonal_pred'].isna().sum()
print("Rows with no seasonal-naive prediction:", missing_seasonal)

seasonal_eval['seasonal_pred'] = seasonal_eval['seasonal_pred'].fillna(global_mean)

seasonal_wmae = wmae(seasonal_eval['y'], seasonal_eval['seasonal_pred'], seasonal_eval['IsHoliday'])
print("Seasonal-naive WMAE:", seasonal_wmae)

Rows with no seasonal-naive prediction: 2033
Seasonal-naive WMAE: 2148.251360661305


In [11]:
import json

baseline_results = {
    "naive_wmae": float(naive_wmae),
    "seasonal_naive_wmae": float(seasonal_wmae),
    "missing_naive_rows": int(missing_naive),
    "missing_seasonal_rows": int(missing_seasonal),
}

with open('baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print(baseline_results)

{'naive_wmae': 2754.5248016741434, 'seasonal_naive_wmae': 2148.251360661305, 'missing_naive_rows': 106, 'missing_seasonal_rows': 2033}


wandb logging

In [12]:
wandb.init(project="walmart-sales-forecasting", name="baselines", group="baseline", job_type="eval")
wandb.log(baseline_results)
wandb.finish()

missing_naive_rows,▁
missing_seasonal_rows,▁
naive_wmae,▁
seasonal_naive_wmae,▁
missing_naive_rows,106
missing_seasonal_rows,2033
naive_wmae,2754.5248
seasonal_naive_wmae,2148.25136
